# Lab 9.11 &mdash; Assemble the Insight Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Bind read-only @tools into a real Groq agent with create_agent
- Withhold any trade/advice tool so it cannot act
- Run it and read the trace where it grounds & cites a figure

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; The financial-report insight agent, end to end](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-11"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Now assemble the insight agent (deck slides 7&ndash;9): `@tool` **read-only** tools (`extract_figure`,
`compute`) bound with **`create_agent`** into a runnable **`CompiledStateGraph`**, driven by the **real**
Groq model. The agent grounds a figure and states it with its source. The design choice: the tools list is
**read-only** &mdash; `place_trade` is **defined but not bound** &mdash; and the output is wrapped
**`needs_review`** for a human analyst. The guardrail is what's **missing** from the tools list.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

from langchain_core.tools import tool

@tool
def extract_figure(name: str) -> dict:
    """Pull a reported figure and its source from the filing. Use this to ground any number before stating it."""
    return REPORT.get(name, {})

@tool
def compute(expression: str) -> str:
    """Do exact arithmetic on a financial expression such as '(120-107.1)/107.1*100'. Use for any calculation."""
    try:
        return str(safe_calc(expression))          # success path
    except Exception:
        return "error: not a numeric expression"    # a tool must NEVER raise -- return a string

@tool
def place_trade(ticker: str, shares: int) -> str:
    """Place a trade. (Defined to show the capability -- but DELIBERATELY WITHHELD from the agent.)"""
    return "TRADED"

print("read-only tools:", extract_figure.name, "&", compute.name, "| withheld:", place_trade.name)

## Build it
Build the agent with **read-only** tools (no `place_trade`), bound to the real `llm`, and wrap whatever
it finds as **`needs_review`**. Run the cell to confirm the wiring &mdash; then run it for real below.

In [ ]:
from langchain.agents import create_agent

def readonly_tools():
    return ___   # TODO: read-only -- [extract_figure, compute], NEVER place_trade

def make_insight_agent():
    return create_agent(llm, ___)   # TODO: bind the read-only tools to the real Groq model

def wrap_needs_review(insight, tools_used):
    # analysis only: wrap the finding so a human analyst reviews it -- never a decision
    return {"insight": insight, "status": ___, "tools_used": tools_used}   # TODO: needs_review

try:
    print("bound tools :", [t.name for t in readonly_tools()])
    print("can it trade?:", "place_trade" in [t.name for t in readonly_tools()])
    print("place_trade still exists as a capability, just unbound:", place_trade.name)
    print("wrapped shape:", wrap_needs_review("Revenue 120.0M [p4].", ["extract_figure"])["status"])
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Assemble the real agent, invoke it on a grounding task, and read the **real trace**: the model calls `extract_figure`, reads the observation, and states revenue **with its page cite** &mdash; no trade tool anywhere.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing. On the free tier, if you hit a rate limit wait a few seconds and re-run._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to .env (free at console.groq.com), then re-run this cell.")
else:
    try:
        agent = make_insight_agent()
        print("agent type:", type(agent).__name__)          # a runnable CompiledStateGraph
        result = agent.invoke({"messages": [{"role": "user",
                 "content": "Use extract_figure to get revenue, then state it with its source. Give NO advice."}]},
                 config={"recursion_limit": 8})
        print_trace(result)
        print("---")
        used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
        out = wrap_needs_review(result["messages"][-1].content, used)
        print("tools used:", out["tools_used"])
        print("status    :", out["status"], "(no trade tool bound, so the agent cannot act)")
        print("insight   :", out["insight"])
    except Exception as e:
        print("(Rate-limited on the free tier? wait a few seconds. Or fill the ___ blanks, then re-run.)", type(e).__name__)

## What to notice
- `type(agent).__name__` is **`CompiledStateGraph`** &mdash; a real runnable graph, driven by the real Groq model.
- The trace shows **`TOOL CALL: extract_figure`** &rarr; an **`OBS:`** with the sourced figure &rarr; a final answer that **cites the page** &mdash; grounding, for real.
- **`place_trade` is never in `readonly_tools()`**, so however the model is prompted it *cannot* trade. The output is `needs_review` for a human.

## Your turn (open task &mdash; no grader)
Ask the agent a question that needs **both** tools &mdash; e.g. *"extract revenue and net income, then use
compute for the net margin, and cite both pages"* &mdash; and re-run. **What good looks like:** the trace
chains `extract_figure` &rarr; `compute`, the answer cites the pages, and there is still no way for it to
trade or advise.

---
### Key takeaway
The guardrail is what's MISSING from the tools list -- place_trade is never bound, so the real agent grounds, computes and cites but cannot trade or advise. Next: run the whole pipeline over a suite.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>